# Day 24 · Paired comparisons: did the change really help?

**First-principles question:** When you change the prompt or the model, how do you know the gain is not luck?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import pandas as pd
from lab.acme import load_cases, load_results
from lab.evals import score
from lab.stats import bootstrap, compare, mcnemar, table

by_id = {c["id"]: c for c in load_cases()}
A, B = "B", "C"          # TODO: pick the two saved runs you want to compare

In [ ]:
def passed(version):
    return {r["id"]: bool(score(by_id[r["id"]], r)["passed"]) for r in load_results(version)}

pa, pb = passed(A), passed(B)
ids = sorted(set(pa) & set(pb))
a, b = [pa[i] for i in ids], [pb[i] for i in ids]
print(len(ids), "cases in both runs")
table(a, b)

## 1 · Paired vs. unpaired
Fill in `bootstrap()` and `mcnemar()` in `lab/stats.py`, then run this cell.

In [ ]:
import importlib, lab.stats
importlib.reload(lab.stats)
from lab.stats import compare
res = compare(a, b)
print(f"{B} − {A} = {res['difference']:+.1%}  (95% {res['low']:+.1%} to {res['high']:+.1%})  McNemar p = {res['p_value']:.3f}")

**Was the change real? Compare with the unpaired error bars from day 7.** …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert table([True, True, False, False], [True, False, True, False]) == {"both": 1, "only_a": 1, "only_b": 1, "neither": 1}
m, lo, hi = lab.stats.bootstrap([False] * 10 + [True] * 10, [True] * 20)
assert abs(m - 0.5) < 1e-9 and lo <= m <= hi, "bootstrap(): mean of b − a, with an interval around it"
assert abs(lab.stats.mcnemar([True] * 10, [False] * 10) - 2 * 0.5 ** 10) < 1e-12, "mcnemar(): exact two-sided p"
assert lab.stats.mcnemar([True, False], [True, False]) == 1.0
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")